In [ ]:
%pip -q install datasets scikit-learn

# Bài 15: Feature Extraction (frozen encoder) vs Fine-Tuning
# Cùng dataset, split, tokenizer, max_length, batch size và epoch để so sánh.
# Learning rate riêng theo chế độ: head-only dùng 1e-3; fine-tuning dùng 2e-5.
import random
import time
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
MODEL_NAME = "bert-base-uncased"
TRAIN_SIZE, VALID_SIZE, TEST_SIZE = 256, 128, 256
MAX_LENGTH = 64
EPOCHS = 1
BATCH_SIZE = 8
LR_FROZEN_HEAD = 1e-3
LR_FINE_TUNE = 2e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading IMDb and preparing fixed splits...")
dataset = load_dataset("stanfordnlp/imdb")
train_val = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE + VALID_SIZE))
train_raw = train_val.select(range(TRAIN_SIZE))
valid_raw = train_val.select(range(TRAIN_SIZE, TRAIN_SIZE + VALID_SIZE))
test_raw = dataset["test"].shuffle(seed=SEED + 1).select(range(TEST_SIZE))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

def prepare_split(split):
    encoded = split.map(tokenize_batch, batched=True, remove_columns=["text"])
    encoded = encoded.rename_column("label", "labels")
    encoded.set_format(type="torch")
    return encoded

train_data = prepare_split(train_raw)
valid_data = prepare_split(valid_raw)
test_data = prepare_split(test_raw)
valid_loader = DataLoader(valid_data, batch_size=16, shuffle=False)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

def evaluate(model, loader):
    model.eval()
    labels, predictions = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model(**batch).logits
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            labels.extend(batch["labels"].cpu().tolist())
    return accuracy_score(labels, predictions), f1_score(labels, predictions, average="binary")

def run_mode(mode):
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    if mode == "frozen":
        for parameter in model.base_model.parameters():
            parameter.requires_grad = False
        learning_rate = LR_FROZEN_HEAD
        mode_name = "Frozen encoder"
    else:
        learning_rate = LR_FINE_TUNE
        mode_name = "Fine-tuning"

    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=learning_rate
    )
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    model.train()
    start = time.perf_counter()
    for epoch in range(EPOCHS):
        for batch in train_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
    training_seconds = time.perf_counter() - start

    valid_accuracy, valid_f1 = evaluate(model, valid_loader)
    test_accuracy, test_f1 = evaluate(model, test_loader)
    result = {
        "mode": mode_name,
        "encoder_trainable": mode != "frozen",
        "learning_rate": learning_rate,
        "trainable_parameters": trainable_parameters,
        "validation_accuracy": valid_accuracy,
        "validation_f1": valid_f1,
        "accuracy": test_accuracy,
        "f1": test_f1,
        "training_seconds": training_seconds,
    }
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

print(f"Device: {device}; model: {MODEL_NAME}; train/validation/test: {TRAIN_SIZE}/{VALID_SIZE}/{TEST_SIZE}; max_length={MAX_LENGTH}; epochs={EPOCHS}; batch_size={BATCH_SIZE}")
results = [run_mode("frozen"), run_mode("finetune")]
print("\nRESULTS ON TEST SET")
for result in results:
    print(f"{result['mode']}: encoder_trainable={result['encoder_trainable']}, Accuracy={result['accuracy']:.4f}, F1={result['f1']:.4f}, training_time={result['training_seconds']:.2f}s, trainable_parameters={result['trainable_parameters']:,}, learning_rate={result['learning_rate']}")
    print(f"  Validation: Accuracy={result['validation_accuracy']:.4f}, F1={result['validation_f1']:.4f}")
print("\nControlled setup: same sampled IMDb splits, tokenizer, architecture, max_length, batch size, optimizer, and epoch count. The frozen mode trains only the classification head; fine-tuning updates the encoder and head.")

Loading IMDb and preparing fixed splits...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (16 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



RESULTS ON TEST SET
Frozen encoder: encoder_trainable=False, Accuracy=0.5586, F1=0.1102, training_time=1.84s, trainable_parameters=1,538, learning_rate=0.001
  Validation: Accuracy=0.4922, F1=0.1558
Fine-tuning: encoder_trainable=True, Accuracy=0.6914, F1=0.5907, training_time=3.39s, trainable_parameters=109,483,778, learning_rate=2e-05
  Validation: Accuracy=0.6719, F1=0.6038

Controlled setup: same sampled IMDb splits, tokenizer, architecture, max_length, batch size, optimizer, and epoch count. The frozen mode trains only the classification head; fine-tuning updates the encoder and head.
